<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-09-multimodal-and-pretrained/lesson-9.1-multimodal/practice/GCP_Capstone_9.1_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 9.1 — Gemini Multimodal

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup — install, authenticate, init the Vertex client

Run this cell first. It installs the SDK, authenticates with Application Default Credentials (no API keys), and creates a Vertex-backed `google-genai` client that every exercise below reuses.

In [ ]:
%%bash
pip install -q google-genai Pillow reportlab

In [ ]:
from google import genai
from google.genai import types
import pathlib, time, json, urllib.request

# Application Default Credentials on Colab (no API keys)
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass  # running outside Colab -> rely on local ADC

PROJECT_ID = 'documind-ai-YOUR-ID'   # <-- replace with your project id
LOCATION   = 'global'                # Gemini 3.x generation is served from the global endpoint
USD_INR    = 85                      # for INR cost displays

# Vertex AI client — NEVER vertexai=True, NEVER an api_key
client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)
print('SDK ready:', genai.__version__)

## Exercise 1: Analyze a receipt image

**Difficulty:** Easy

Send a photo of a receipt to Gemini. Extract vendor, date, total as JSON.

1. Download (or read) a receipt image and load its bytes.
2. Send the bytes as a `types.Part.from_bytes(...)` alongside a text instruction.
3. Force JSON output with `response_mime_type='application/json'` and a low temperature.
4. Print the structured result.

In [ ]:
# Download a sample receipt image
url = 'https://upload.wikimedia.org/wikipedia/commons/0/0b/ReceiptSwiss.jpg'
_req = urllib.request.Request(url, headers={'User-Agent': 'DocuMind-Capstone/1.0 (learning)'})
with urllib.request.urlopen(_req, timeout=30) as _r:  # Wikimedia 403s the default urllib UA
    pathlib.Path('receipt.jpg').write_bytes(_r.read())

img_bytes = pathlib.Path('receipt.jpg').read_bytes()
print(f'Image size: {len(img_bytes)//1024} KB')

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=[
        types.Part.from_bytes(data=img_bytes, mime_type='image/jpeg'),
        'Extract all text from this receipt. Return vendor_name, date, items, total as JSON.'
    ],
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        temperature=0.1,
    ),
)
print(response.text)

## Exercise 2: Process a multi-page PDF

**Difficulty:** Easy

Upload a PDF with tables. Extract table data as structured JSON.

1. Create (or load) a PDF that contains invoice/table data.
2. Read the PDF bytes and pass them as a `Part.from_bytes(..., mime_type='application/pdf')` — Gemini reads PDFs natively, no OCR step needed.
3. Ask for the table rows back as JSON.
4. Print the structured result.

In [ ]:
# Create a sample PDF with tabular invoice data
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

c = canvas.Canvas('sample.pdf', pagesize=letter)
c.drawString(72, 700, 'Invoice #2024-001')
c.drawString(72, 680, 'Vendor: TechCorp Solutions')
c.drawString(72, 660, 'Date: 2024-03-15')
c.drawString(72, 640, 'Item: Cloud Services - $4,500')
c.drawString(72, 620, 'Item: Support Package - $1,200')
c.drawString(72, 600, 'Total: $5,700')
c.save()
print('PDF created')

# Native PDF vision — extract the table rows as JSON
pdf_bytes = pathlib.Path('sample.pdf').read_bytes()
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=[
        types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
        'Extract every line item as a JSON array of {item, amount} rows, plus a total field.'
    ],
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        temperature=0.1,
    ),
)
print(response.text)

## Exercise 3: Count tokens for different modalities

**Difficulty:** Easy

Use `count_tokens()` for an image, PDF, and text prompt. Compare costs.

1. Call `client.models.count_tokens(...)` on the image + prompt from Exercise 1.
2. Estimate token counts for other modalities (image, PDF pages, audio minutes, video minutes).
3. Multiply by the Flash input price (USD 1.50 / 1M tokens) to get a per-item cost.
4. Print counts and costs so image > text is visible.

In [ ]:
# Real token count for the receipt image + prompt
token_count = client.models.count_tokens(
    model='gemini-3.6-flash',
    contents=[
        types.Part.from_bytes(data=img_bytes, mime_type='image/jpeg'),
        'Extract text',
    ],
)
FLASH_IN = 1.50  # USD per 1M input tokens (course standard rate)
print(f'Image + prompt tokens: {token_count.total_tokens}')
print(f'Estimated cost (Flash): ${token_count.total_tokens * FLASH_IN / 1_000_000:.6f}')

# Rule-of-thumb token estimates per modality
def estimate_tokens(modality, value):
    if modality == 'image_small':   return 258
    if modality == 'image_1024':    return 1290
    if modality == 'pdf_pages':     return value * 258
    if modality == 'audio_minutes': return value * 1920
    if modality == 'video_minutes': return value * 18000

print('\nmodality            tokens        USD        INR')
for name, val in [('image_small', 1), ('image_1024', 1), ('pdf_pages', 50),
                  ('audio_minutes', 30), ('video_minutes', 5)]:
    tokens = estimate_tokens(name, val)
    cost = tokens * FLASH_IN / 1_000_000
    print(f'{name+"("+str(val)+")":<20}{tokens:>10,}   ${cost:>8.4f}   Rs.{cost*USD_INR:>8.4f}')

## Exercise 4: Audio transcription with speakers

**Difficulty:** Medium

Upload an MP3 and transcribe it with speaker labels and timestamps.

1. Point at an audio file staged in Cloud Storage (Vertex reads large media from GCS via `Part.from_uri`, not the Developer-API File-upload endpoint).
2. Reference it with `types.Part.from_uri(file_uri='gs://...', mime_type='audio/mpeg')`.
3. Prompt for a transcript in `[Speaker N MM:SS]` format, followed by a bulleted action-items list.
4. Print the transcript.

> Replace `gs://documind-ai-YOUR-ID-media/meeting.mp3` with your own uploaded audio (`gsutil cp meeting.mp3 gs://...`).

In [ ]:
# Vertex reads large audio/video from GCS via Part.from_uri (the File API upload
# endpoint is Developer-API-only; on Vertex you stage media in a bucket instead).
# PREREQUISITE: stage your own audio/video in a GCS bucket first, e.g.
#   gsutil mb -l us-central1 gs://documind-ai-YOUR-ID-media
#   gsutil cp meeting.mp3 review.mp4 gs://documind-ai-YOUR-ID-media/
# then point AUDIO_URI / VIDEO_URI at your objects (Gemini reads them via Part.from_uri).
AUDIO_URI = 'gs://documind-ai-YOUR-ID-media/meeting.mp3'  # <-- your uploaded MP3

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=[
        types.Part.from_uri(file_uri=AUDIO_URI, mime_type='audio/mpeg'),
        'Transcribe this meeting. Label each turn as [Speaker N MM:SS]. '
        'After the transcript, list the action items as a bulleted list.'
    ],
)
print(response.text)

## Exercise 5: Video timestamp queries

**Difficulty:** Medium

Upload a video and ask about events at specific timestamps.

1. Stage the video in GCS and reference it with `Part.from_uri(..., mime_type='video/mp4')`.
2. Ask targeted questions about specific timestamps — Gemini reasons over both the visual frames and the audio track.
3. Print the descriptions returned for each requested timestamp.

In [ ]:
VIDEO_URI = 'gs://documind-ai-YOUR-ID-media/review.mp4'  # <-- your uploaded MP4

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=[
        types.Part.from_uri(file_uri=VIDEO_URI, mime_type='video/mp4'),
        'For the timestamps 00:15, 01:30 and 03:00, describe both what is shown '
        'on screen and what is said in the audio at that moment.'
    ],
)
print(response.text)

## Exercise 6: Batch invoice processing

**Difficulty:** Medium

Send 5-10 invoice images in ONE call. Extract all as a JSON array.

1. Collect several image byte blobs.
2. Build a single `contents` list with one `Part.from_bytes(...)` per image, followed by the instruction.
3. Request a JSON array back with `response_mime_type='application/json'`.
4. Parse and print the array — one object per invoice.

In [ ]:
# Stand-in batch: reuse the receipt bytes a few times so the cell runs top-to-bottom.
# In production, read each real invoice file's bytes here instead.
batch_images = [img_bytes, img_bytes, img_bytes]  # e.g. 5-10 distinct invoices

contents = [types.Part.from_bytes(data=b, mime_type='image/jpeg') for b in batch_images]
contents.append(
    'Each image above is a separate invoice. Return a JSON array where every '
    'element is {invoice_index, vendor_name, date, total} for one invoice, in order.'
)

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=contents,
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        temperature=0.1,
    ),
)
invoices = json.loads(response.text)
print(f'Extracted {len(invoices)} invoices')
print(json.dumps(invoices, indent=2))

## Exercise 7: Cross-modal analysis

**Difficulty:** Challenge

Send invoice image + contract PDF + meeting audio in one call. Cross-reference for discrepancies.

1. Assemble a single `contents` list mixing an image `Part.from_bytes`, a PDF `Part.from_bytes`, and an audio `Part.from_uri`.
2. Ask Gemini to reconcile them: does the invoiced amount match the contract terms, and was it approved in the meeting?
3. Request structured JSON with `invoice_summary`, `contract_terms`, `approvals`, and `discrepancies`.
4. Print the reconciliation.

In [ ]:
# Cross-modal reconciliation: image bytes + PDF bytes + audio from GCS in ONE call.
contents = [
    'INVOICE (image):',
    types.Part.from_bytes(data=img_bytes, mime_type='image/jpeg'),
    'CONTRACT (pdf):',
    types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
    'APPROVAL MEETING (audio):',
    types.Part.from_uri(file_uri=AUDIO_URI, mime_type='audio/mpeg'),
    'Cross-reference these three sources. Return JSON with keys: '
    'invoice_summary, contract_terms, approvals (who approved what, from the audio), '
    'and discrepancies (a list of any mismatches between invoice, contract and approvals).'
]

response = client.models.generate_content(
    model='gemini-3.1-pro-preview',  # high-reasoning model for cross-modal reconciliation
    contents=contents,
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        temperature=0.2,
        thinking_config=types.ThinkingConfig(thinking_budget=2048),
    ),
)
print(response.text)

## Exercise 8: Format validation pipeline

**Difficulty:** Challenge

Build an ingestion step that validates formats, converts unsupported types, and routes by file size.

1. Map each incoming MIME type to a supported Gemini modality; reject or convert anything unsupported.
2. Route by size: small files go inline as `Part.from_bytes`; large media (audio/video, or over the inline threshold) are staged in GCS and referenced with `Part.from_uri`.
3. Wrap the call in the retry helper (backoff on 429/503, no retry on 400).
4. Run the pipeline over a few sample inputs and print the routing decision + result.

In [ ]:
import random
from google.genai import errors

# --- retry helper (lifted from the lesson's error-handling cell) ---
def analyze_with_retry(contents, model='gemini-3.6-flash', config=None, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=model, contents=contents, config=config)
        except errors.APIError as e:
            if e.code == 429:
                wait = (2 ** attempt) + random.uniform(0, 1)
                print(f'  Rate limited. Waiting {wait:.1f}s...'); time.sleep(wait)
            elif e.code == 503:
                print('  Service unavailable. Waiting 10s...'); time.sleep(10)
            elif e.code == 400:
                print(f'  Invalid input (no retry): {e.message}'); raise
            else:
                raise
    raise RuntimeError('Max retries exceeded')

# --- format validation + routing ---
SUPPORTED = {
    'image/jpeg': 'image', 'image/png': 'image',
    'application/pdf': 'document',
    'audio/mpeg': 'audio', 'audio/wav': 'audio',
    'video/mp4': 'video',
}
INLINE_LIMIT = 7 * 1024 * 1024  # ~7MB: below this we can send bytes inline

def build_part(source):
    """source: dict with mime_type and either 'bytes' or 'gcs_uri'."""
    mime = source['mime_type']
    if mime not in SUPPORTED:
        raise ValueError(f'Unsupported format: {mime}')
    modality = SUPPORTED[mime]
    # audio/video always go via GCS; everything else routes by size
    if modality in ('audio', 'video') or source.get('gcs_uri'):
        uri = source.get('gcs_uri')
        if not uri:
            raise ValueError(f'{modality} must be staged in GCS (provide gcs_uri)')
        print(f'  route: Part.from_uri  ({modality})')
        return types.Part.from_uri(file_uri=uri, mime_type=mime)
    data = source['bytes']
    if len(data) > INLINE_LIMIT:
        raise ValueError(f'{len(data)//1024}KB exceeds inline limit; stage in GCS')
    print(f'  route: Part.from_bytes ({modality}, {len(data)//1024}KB)')
    return types.Part.from_bytes(data=data, mime_type=mime)

# --- run over sample inputs ---
inputs = [
    {'name': 'receipt.jpg',  'mime_type': 'image/jpeg',      'bytes': img_bytes},
    {'name': 'sample.pdf',   'mime_type': 'application/pdf',  'bytes': pdf_bytes},
    {'name': 'meeting.mp3',  'mime_type': 'audio/mpeg',       'gcs_uri': AUDIO_URI},
    {'name': 'notes.txt',    'mime_type': 'text/plain',       'bytes': b'unsupported'},
]

for item in inputs:
    print(f"\n{item['name']}:")
    try:
        part = build_part(item)
    except ValueError as e:
        print(f'  skipped -> {e}')
        continue
    resp = analyze_with_retry([part, 'Summarize this document in one sentence.'])
    print(f'  result: {resp.text.strip()[:120]}')